# 02 - Preparación de los datos y entrenamiento de modelos

**Proyecto de aula - Modelos y Simulación de Sistemas II**

**Integrantes:** Kelly Julieth Arango Henao, Lisbeth Cataño Sepulveda

## Introducción

En este notebook se prepara el conjunto de datos y se entrenan y comparan los cinco modelos del proyecto:

| Tipo de modelo | Modelo |
|---|---|
| Paramétrico | Regresión logística |
| No paramétrico | KNN |
| Ensamble de árboles | Random Forest |
| Red neuronal | Perceptrón multicapa (MLP) |
| Máquina de vectores de soporte | SVM |

Los pasos son:

1. Cargar los datos y eliminar los duplicados (igual que en el notebook 01).
2. Separar las variables de entrada de la variable objetivo.
3. Dividir los datos en entrenamiento y prueba.
4. Definir el preprocesamiento (estandarizar solo `Time` y `Amount`).
5. Entrenar cada modelo con validación cruzada y revisar el efecto de sus hiperparámetros.
6. Comparar los cinco modelos en una sola tabla.

El conjunto de prueba **no se usa en este notebook**. Se deja aparte hasta que estén definidos los dos mejores modelos.

## 0. Antes de ejecutar este notebook

Igual que en el notebook 01, este código busca el archivo `creditcard.csv` en las ubicaciones habituales (carpeta `data/` del proyecto o Google Drive en Colab). Si no lo encuentra, pide cargarlo manualmente.

In [ ]:
import os

RUTA_DATOS = None

candidatos = [
    "../data/creditcard.csv",
    "data/creditcard.csv",
    "/content/drive/MyDrive/creditcard.csv",
    "/content/drive/MyDrive/data/creditcard.csv",
    "/content/drive/MyDrive/Colab Notebooks/data/creditcard.csv",
    "creditcard.csv"
]

for ruta in candidatos:
    if os.path.exists(ruta):
        RUTA_DATOS = ruta
        break

if RUTA_DATOS is None:
    try:
        import google.colab
        en_colab = True
    except ImportError:
        en_colab = False

    if en_colab:
        from google.colab import drive

        print("No se encontró el archivo. Conectando Google Drive...")
        drive.mount('/content/drive')

        for ruta in candidatos:
            if os.path.exists(ruta):
                RUTA_DATOS = ruta
                break

        if RUTA_DATOS is None:
            from google.colab import files
            print("Selecciona el archivo creditcard.csv")
            subido = files.upload()

            if subido:
                RUTA_DATOS = list(subido.keys())[0]

if RUTA_DATOS is None:
    raise FileNotFoundError(
        "No se encontró creditcard.csv. Revisa la carpeta data/ "
        "o consulta el README para descargar el archivo."
    )

print("Archivo de datos utilizado:", RUTA_DATOS)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.utils.metaestimators import available_if

sns.set_style("whitegrid")

# Semilla fija para que los resultados se puedan repetir
SEMILLA = 42

# Carpeta para guardar tablas y gráficas
CARPETA_RESULTADOS = (
    "../resultados"
    if os.path.isdir("..") and os.path.isdir("../notebooks")
    else "resultados"
)
os.makedirs(CARPETA_RESULTADOS, exist_ok=True)

print("Carpeta de resultados:", CARPETA_RESULTADOS)

## 1. Cargar y limpiar los datos

Se cargan los datos y se eliminan los registros duplicados **antes** de dividir en entrenamiento y prueba. Si se dividiera primero, una transacción repetida podría quedar una vez en entrenamiento y otra en prueba, y el modelo parecería mejor de lo que realmente es.

In [ ]:
df = pd.read_csv(RUTA_DATOS)
filas_originales = len(df)

df = df.drop_duplicates().reset_index(drop=True)

print("Filas originales:", filas_originales)
print("Filas después de eliminar duplicados:", len(df))
print("Duplicados eliminados:", filas_originales - len(df))
print("Fraudes:", int(df["Class"].sum()), f"({df['Class'].mean() * 100:.3f} %)")

## 2. Variables de entrada (X) y variable objetivo (y)

`Class` es lo que se quiere predecir, así que no puede estar entre las variables de entrada. Si estuviera, el modelo tendría la respuesta y no aprendería nada.

In [ ]:
X = df.drop(columns="Class")
y = df["Class"]

print("Variables de entrada:", X.shape[1])
print("Columnas:", X.columns.tolist())

## 3. División en entrenamiento y prueba

Se separa el 80 % de los datos para entrenamiento y el 20 % para prueba. La división es **estratificada**: entrenamiento y prueba conservan la misma proporción de fraudes que el conjunto completo. Sin esto, con tan pocos fraudes, el conjunto de prueba podría quedar con muy pocos o con demasiados.

La semilla fija (`random_state=42`) permite repetir exactamente la misma división.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=SEMILLA
)

resumen_particion = pd.DataFrame({
    "Transacciones": [len(y_train), len(y_test)],
    "Fraudes": [int(y_train.sum()), int(y_test.sum())],
    "% de fraude": [y_train.mean() * 100, y_test.mean() * 100]
}, index=["Entrenamiento", "Prueba"]).round(3)

resumen_particion

Con pocos fraudes en el conjunto de prueba, las métricas que se calculen sobre él tendrán intervalos de confianza amplios. Por eso en el reporte se presentan junto con su intervalo.

## 4. Preprocesamiento

- `Time` y `Amount` se estandarizan, porque están en escalas muy distintas a las demás.
- `V1` a `V28` **no se estandarizan de nuevo**: ya son componentes de un PCA y vienen en una escala comparable.

El estandarizador se mete dentro de un `Pipeline`. Así, en cada partición de la validación cruzada se calcula la media y la desviación **solo con los datos de entrenamiento de esa partición**, y no se filtra información de los datos de validación.

In [ ]:
columnas_a_escalar = ["Time", "Amount"]

preprocesamiento = ColumnTransformer(
    transformers=[("escalar", StandardScaler(), columnas_a_escalar)],
    remainder="passthrough"   # V1-V28 pasan sin cambios
)

## 5. Manejo del desbalance

Con solo 0,17 % de fraudes, generar muchos ejemplos sintéticos con SMOTE puede deformar la zona donde se separan las clases. Por eso se empieza con la opción más sencilla: **ponderar las clases** (`class_weight="balanced"`). Así, equivocarse en un fraude pesa más que equivocarse en una transacción legítima. Más adelante se pueden comparar otras alternativas (submuestreo, ADASYN).

**Referencia para la métrica principal.** El AUPRC (área bajo la curva Precision-Recall) de un modelo que no sabe nada es igual a la proporción de fraudes. Cualquier modelo útil tiene que superar ese valor.

In [ ]:
referencia_auprc = y_train.mean()
print(f"AUPRC de un modelo sin información (proporción de fraudes): {referencia_auprc:.4f}")

## 6. Validación cruzada

Se usa validación cruzada estratificada de 5 particiones sobre el conjunto de entrenamiento. En cada vuelta, 4 partes se usan para entrenar y 1 para validar, y cada parte conserva la proporción de fraudes.

Se calculan varias métricas. El AUPRC es la principal; el resto sirve para entender mejor el comportamiento del modelo:

- **AUPRC** (`average_precision`): resume el balance entre precision y recall sin depender de un umbral.
- **ROC-AUC:** capacidad general de separar las dos clases.
- **Precision:** de las transacciones que el modelo marca como fraude, cuántas lo son.
- **Recall:** de los fraudes reales, cuántos detecta el modelo.
- **F1:** combina precision y recall.
- **MCC:** coeficiente de correlación de Matthews, definido en el reporte.

In [ ]:
K = 5
validacion = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEMILLA)

metricas = {
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "mcc": "matthews_corrcoef",
}

## 7. Regresión logística

Es el modelo paramétrico del proyecto. Su hiperparámetro principal es `C`, que controla qué tanto se penalizan los coeficientes grandes: un `C` pequeño regulariza fuerte (modelo más simple) y un `C` grande regulariza poco.

Se prueba una malla de valores de `C` y se elige el que da mejor AUPRC en validación.

In [ ]:
modelo_lr = Pipeline([
    ("prep", preprocesamiento),
    ("clf", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=SEMILLA
    ))
])

malla_lr = {"clf__C": [0.001, 0.01, 0.1, 1, 10, 100]}

busqueda_lr = GridSearchCV(
    estimator=modelo_lr,
    param_grid=malla_lr,
    scoring=metricas,
    refit="average_precision",
    cv=validacion,
    return_train_score=True,
    n_jobs=-1
)

busqueda_lr.fit(X_train, y_train)

print("Mejor valor de C:", busqueda_lr.best_params_["clf__C"])
print(f"Mejor AUPRC en validación: {busqueda_lr.best_score_:.4f}")

## 8. Efecto del hiperparámetro `C`

La tabla y la gráfica muestran cómo cambia el desempeño al variar `C`, tanto en entrenamiento como en validación. Si el desempeño en entrenamiento es mucho mayor que en validación, el modelo se está sobreajustando.

In [ ]:
res = busqueda_lr.cv_results_

tabla_C = pd.DataFrame({
    "C": [float(c) for c in res["param_clf__C"]],
    "AUPRC entrenamiento": res["mean_train_average_precision"],
    "AUPRC validación": res["mean_test_average_precision"],
    "Desv. AUPRC validación": res["std_test_average_precision"],
    "ROC-AUC validación": res["mean_test_roc_auc"],
    "Recall validación": res["mean_test_recall"],
    "Precision validación": res["mean_test_precision"],
}).round(4)

tabla_C.to_csv(f"{CARPETA_RESULTADOS}/regresion_logistica_efecto_C.csv", index=False)
tabla_C

In [ ]:
plt.figure(figsize=(7, 4))

plt.semilogx(tabla_C["C"], tabla_C["AUPRC entrenamiento"], marker="o", label="Entrenamiento")
plt.semilogx(tabla_C["C"], tabla_C["AUPRC validación"], marker="s", linestyle="--", label="Validación")
plt.ylim(0.70, 0.80)

plt.xlabel("Hiperparámetro C (escala logarítmica)")
plt.ylabel("AUPRC")
plt.title("Regresión logística: efecto de C en el AUPRC")
plt.legend()
plt.tight_layout()

plt.savefig(f"{CARPETA_RESULTADOS}/regresion_logistica_efecto_C.png", dpi=200)
plt.show()

## 9. Resultados del mejor modelo con intervalos de confianza

Para cada métrica se calcula el promedio de las 5 particiones y un intervalo de confianza del 95 % (distribución t de Student, con 4 grados de libertad). Como las particiones comparten datos de entrenamiento, el intervalo es aproximado, pero sirve para ver qué tan estable es el resultado.

In [ ]:
def resumen_con_ic(cv_res, indice, nombre_metrica, conjunto, k=K):
    """Promedio e IC 95 % de una métrica a partir de los valores de cada partición."""
    valores = np.array([cv_res[f"split{i}_{conjunto}_{nombre_metrica}"][indice] for i in range(k)])
    media = valores.mean()
    desv = valores.std(ddof=1)
    mitad = stats.t.ppf(0.975, k - 1) * desv / np.sqrt(k)
    return media, mitad

indice_mejor = busqueda_lr.best_index_

nombres_legibles = {
    "average_precision": "AUPRC",
    "roc_auc": "ROC-AUC",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1-score",
    "mcc": "MCC",
}

filas = []
for nombre in metricas:
    m_tr, h_tr = resumen_con_ic(res, indice_mejor, nombre, "train")
    m_va, h_va = resumen_con_ic(res, indice_mejor, nombre, "test")   # "test" aquí es la partición de validación
    filas.append({
        "Métrica": nombres_legibles[nombre],
        "Entrenamiento": f"{m_tr:.4f} ± {h_tr:.4f}",
        "Validación": f"{m_va:.4f} ± {h_va:.4f}",
    })

tabla_mejor = pd.DataFrame(filas)
tabla_mejor.to_csv(f"{CARPETA_RESULTADOS}/regresion_logistica_mejor_modelo.csv", index=False)

print("Regresión logística con C =", busqueda_lr.best_params_["clf__C"])
tabla_mejor

### Interpretación

Con `class_weight="balanced"` y el umbral de decisión por defecto (0,5), el modelo marca como fraude muchas transacciones que no lo son. Por eso se ve un **recall alto y una precision baja**. El AUPRC no depende del umbral, así que es la métrica más justa para comparar modelos.

- Mejor valor de `C`: 0,1. Entre 0,01 y 100 el AUPRC casi no cambia, así que la regularización tiene poco efecto en este problema.
- AUPRC en validación: 0,7545, frente a 0,0017 de un modelo sin información.
- Entrenamiento y validación: 0,7568 y 0,7545. Son casi iguales, por lo que no hay sobreajuste.
- Precision y recall: recall de 0,9153 y precision de 0,0605. El modelo detecta cerca del 91 % de los fraudes, pero por cada fraude real marca unas 15 transacciones legítimas como fraude.

## 10. Herramientas para comparar los modelos

Antes de entrenar los demás modelos se definen tres herramientas que se reutilizan.

**Submuestreo de la clase mayoritaria.** KNN y la red neuronal no tienen ponderación de clases, y la SVM con 227 mil filas es muy lenta. Para estos tres modelos se usa submuestreo: en cada entrenamiento se conservan todos los fraudes y solo una muestra aleatoria de transacciones legítimas (20 por cada fraude). Esto se hace **únicamente con los datos de entrenamiento de cada partición**; los datos de validación se dejan completos y con su proporción real de fraudes, así que las métricas siguen siendo comparables entre modelos.

La regresión logística y el Random Forest usan todos los datos con ponderación de clases.

In [ ]:
RELACION_SUBMUESTREO = 20   # transacciones legítimas conservadas por cada fraude


class ConSubmuestreo(ClassifierMixin, BaseEstimator):
    """Envuelve un clasificador y submuestrea la clase mayoritaria SOLO al entrenar.

    Al predecir no modifica los datos, por eso se puede usar dentro de la
    validación cruzada sin que los datos de validación se alteren.
    """

    def __init__(self, estimador=None, relacion=RELACION_SUBMUESTREO, semilla=SEMILLA):
        self.estimador = estimador
        self.relacion = relacion
        self.semilla = semilla

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        rng = np.random.default_rng(self.semilla)

        idx_fraude = np.where(y == 1)[0]
        idx_legit = np.where(y == 0)[0]
        n_legit = min(len(idx_legit), self.relacion * len(idx_fraude))
        idx_legit = rng.choice(idx_legit, size=n_legit, replace=False)

        idx = np.concatenate([idx_fraude, idx_legit])
        rng.shuffle(idx)

        self.estimador_ = clone(self.estimador).fit(X[idx], y[idx])
        self.classes_ = self.estimador_.classes_
        return self

    def predict(self, X):
        return self.estimador_.predict(X)

    @available_if(lambda self: hasattr(self.estimador, "predict_proba"))
    def predict_proba(self, X):
        return self.estimador_.predict_proba(X)

    @available_if(lambda self: hasattr(self.estimador, "decision_function"))
    def decision_function(self, X):
        return self.estimador_.decision_function(X)

In [ ]:
resultados_modelos = {"Regresión logística": busqueda_lr}
mallas_usadas = {"Regresión logística": malla_lr}


def entrenar_modelo(nombre, modelo, malla):
    """Entrena un modelo con la búsqueda de hiperparámetros y la validación cruzada
    definidas arriba, y devuelve la búsqueda y la tabla con todas las combinaciones."""
    busqueda = GridSearchCV(
        estimator=modelo,
        param_grid=malla,
        scoring=metricas,
        refit="average_precision",
        cv=validacion,
        return_train_score=True,
        n_jobs=-1
    )

    inicio = time.time()
    busqueda.fit(X_train, y_train)
    duracion = time.time() - inicio

    r = busqueda.cv_results_

    # Una columna por hiperparámetro (se quita el prefijo del pipeline)
    parametros = pd.DataFrame(r["params"])
    parametros.columns = [c.split("__")[-1] for c in parametros.columns]
    parametros = parametros.astype(str)

    tabla = parametros.copy()
    tabla["AUPRC entrenamiento"] = r["mean_train_average_precision"]
    tabla["AUPRC validación"] = r["mean_test_average_precision"]
    tabla["Desv. AUPRC validación"] = r["std_test_average_precision"]
    tabla["Recall validación"] = r["mean_test_recall"]
    tabla["Precision validación"] = r["mean_test_precision"]
    tabla = tabla.round(4)

    archivo = nombre.lower().replace(" ", "_").replace("ó", "o").replace("í", "i")
    tabla.to_csv(f"{CARPETA_RESULTADOS}/{archivo}_efecto_hiperparametros.csv", index=False)

    resultados_modelos[nombre] = busqueda
    mallas_usadas[nombre] = malla

    print(f"{nombre}: búsqueda terminada en {duracion / 60:.1f} minutos")
    print("Mejores hiperparámetros:", {k.split("__")[-1]: v for k, v in busqueda.best_params_.items()})
    print(f"Mejor AUPRC en validación: {busqueda.best_score_:.4f}")
    return busqueda, tabla


def grafica_efecto(tabla, x, grupo, titulo, archivo, x_log=False):
    """AUPRC en entrenamiento (línea continua) y validación (punteada) en función
    de un hiperparámetro, con una línea por cada valor de otro hiperparámetro."""
    plt.figure(figsize=(7, 4))
    colores = sns.color_palette("tab10")

    for i, valor in enumerate(tabla[grupo].unique()):
        sub = tabla[tabla[grupo] == valor]
        try:
            sub = sub.assign(_x=sub[x].astype(float)).sort_values("_x")
            xs = sub["_x"]
        except ValueError:
            xs = sub[x]
        color = colores[i % len(colores)]
        plt.plot(xs, sub["AUPRC entrenamiento"], marker="o", color=color, label=f"{grupo} = {valor} (entrenamiento)")
        plt.plot(xs, sub["AUPRC validación"], marker="s", linestyle="--", color=color, label=f"{grupo} = {valor} (validación)")

    if x_log:
        plt.xscale("log")
    plt.xlabel(x)
    plt.ylabel("AUPRC")
    plt.title(titulo)
    plt.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(f"{CARPETA_RESULTADOS}/{archivo}.png", dpi=200)
    plt.show()

## 11. KNN (modelo no paramétrico)

KNN clasifica una transacción según las `k` transacciones más parecidas del entrenamiento. Los hiperparámetros que se prueban son:

- `n_neighbors` (k): cuántos vecinos se consultan. Con k pequeño el modelo es más sensible al ruido; con k grande es más suave.
- `weights`: si todos los vecinos pesan igual (`uniform`) o los más cercanos pesan más (`distance`).

Se usa submuestreo de la clase mayoritaria, porque KNN tiene que comparar cada transacción de validación contra todas las de entrenamiento y con 227 mil filas sería muy lento. `V1` a `V28` se dejan sin reescalar, como se explicó en la sección 4.

**Ojo al leer el AUPRC de entrenamiento en KNN.** Al evaluar sobre los mismos datos con los que se entrenó, cada transacción se encuentra a sí misma como su vecino más cercano, por lo que ese valor sale artificialmente alto (sobre todo con `weights="distance"`). No indica sobreajuste real: el valor que sirve para comparar es el de validación.

In [ ]:
modelo_knn = Pipeline([
    ("prep", preprocesamiento),
    ("clf", ConSubmuestreo(estimador=KNeighborsClassifier()))
])

malla_knn = {
    "clf__estimador__n_neighbors": [3, 5, 11, 21, 51],
    "clf__estimador__weights": ["uniform", "distance"],
}

busqueda_knn, tabla_knn = entrenar_modelo("KNN", modelo_knn, malla_knn)
tabla_knn

In [ ]:
grafica_efecto(
    tabla_knn, x="n_neighbors", grupo="weights",
    titulo="KNN: efecto de k y de la ponderación en el AUPRC",
    archivo="knn_efecto_hiperparametros"
)

## 12. Random Forest (ensamble de árboles)

Random Forest entrena muchos árboles de decisión sobre muestras distintas de los datos y combina sus votos. Los hiperparámetros que se prueban son:

- `n_estimators`: número de árboles.
- `max_depth`: profundidad máxima de cada árbol. Árboles muy profundos tienden a memorizar el entrenamiento.

Usa todos los datos con `class_weight="balanced_subsample"`, que pondera las clases dentro de cada árbol. Para que el tiempo sea manejable con 227 mil filas, cada árbol se entrena con una muestra aleatoria del 30 % de los datos (`max_samples=0.3`), distinta para cada árbol, de modo que el bosque completo sigue usando toda la información. Los valores de `max_samples` (0,3) y `min_samples_leaf` (5) se dejan fijos. Es el modelo que más tarda de los cinco.

In [ ]:
modelo_rf = Pipeline([
    ("prep", preprocesamiento),
    ("clf", RandomForestClassifier(
        class_weight="balanced_subsample",
        max_samples=0.3,
        min_samples_leaf=5,
        n_jobs=1,
        random_state=SEMILLA
    ))
])

malla_rf = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [8, 14],
}

busqueda_rf, tabla_rf = entrenar_modelo("Random Forest", modelo_rf, malla_rf)
tabla_rf

In [ ]:
grafica_efecto(
    tabla_rf, x="max_depth", grupo="n_estimators",
    titulo="Random Forest: efecto de la profundidad y del número de árboles en el AUPRC",
    archivo="random_forest_efecto_hiperparametros"
)

## 13. Red neuronal (MLP)

El perceptrón multicapa aprende combinaciones no lineales de las variables mediante capas de neuronas. Los hiperparámetros que se prueban son:

- `hidden_layer_sizes`: número de neuronas en cada capa oculta, por ejemplo `(32,)` es una capa de 32 neuronas y `(64, 32)` son dos capas.
- `alpha`: regularización. Un valor mayor limita el tamaño de los pesos y reduce el sobreajuste.

Usa submuestreo de la clase mayoritaria, porque `MLPClassifier` no permite ponderar las clases. El entrenamiento se detiene solo cuando deja de mejorar (`early_stopping`).

In [ ]:
modelo_mlp = Pipeline([
    ("prep", preprocesamiento),
    ("clf", ConSubmuestreo(estimador=MLPClassifier(
        activation="relu",
        max_iter=500,
        early_stopping=True,
        random_state=SEMILLA
    )))
])

malla_mlp = {
    "clf__estimador__hidden_layer_sizes": [(16,), (32,), (64,), (64, 32)],
    "clf__estimador__alpha": [0.0001, 0.01],
}

busqueda_mlp, tabla_mlp = entrenar_modelo("Red neuronal", modelo_mlp, malla_mlp)
tabla_mlp

In [ ]:
grafica_efecto(
    tabla_mlp, x="hidden_layer_sizes", grupo="alpha",
    titulo="Red neuronal: efecto de la arquitectura y de alpha en el AUPRC",
    archivo="red_neuronal_efecto_hiperparametros"
)

## 14. Máquina de vectores de soporte (SVM)

La SVM busca la frontera que mejor separa las dos clases. Con un kernel lineal la frontera es un plano; con el kernel RBF puede ser curva. Los hiperparámetros que se prueban son:

- `kernel`: lineal o RBF.
- `C`: qué tanto se penalizan los errores en el entrenamiento. Un valor mayor ajusta más a los datos.

Con el kernel RBF se deja `gamma="scale"`. Usa submuestreo de la clase mayoritaria, porque el costo de la SVM crece muy rápido con el número de filas y con 227 mil sería demasiado lenta.

In [ ]:
modelo_svm = Pipeline([
    ("prep", preprocesamiento),
    ("clf", ConSubmuestreo(estimador=SVC(gamma="scale")))
])

malla_svm = {
    "clf__estimador__kernel": ["linear", "rbf"],
    "clf__estimador__C": [0.1, 1, 10],
}

busqueda_svm, tabla_svm = entrenar_modelo("SVM", modelo_svm, malla_svm)
tabla_svm

In [ ]:
grafica_efecto(
    tabla_svm, x="C", grupo="kernel",
    titulo="SVM: efecto de C y del kernel en el AUPRC",
    archivo="svm_efecto_hiperparametros",
    x_log=True
)

## 15. Comparación de los cinco modelos

Se resume el mejor resultado de cada modelo en validación, con el intervalo de confianza del 95 % calculado a partir de las 5 particiones. También se muestra el tiempo promedio que tardó en ajustarse cada modelo con sus mejores hiperparámetros.

In [ ]:
filas = []
for nombre, busqueda in resultados_modelos.items():
    r = busqueda.cv_results_
    i = busqueda.best_index_

    m_tr, _ = resumen_con_ic(r, i, "average_precision", "train")
    m_va, h_va = resumen_con_ic(r, i, "average_precision", "test")

    fila = {
        "Modelo": nombre,
        "Mejores hiperparámetros": ", ".join(f"{k.split('__')[-1]}={v}" for k, v in busqueda.best_params_.items()),
        "AUPRC entrenamiento": round(m_tr, 4),
        "AUPRC validación": round(m_va, 4),
        "IC 95 % (±)": round(h_va, 4),
    }
    for clave, etiqueta in [("roc_auc", "ROC-AUC"), ("precision", "Precision"),
                            ("recall", "Recall"), ("f1", "F1-score"), ("mcc", "MCC")]:
        fila[f"{etiqueta} validación"] = round(r[f"mean_test_{clave}"][i], 4)
    fila["Tiempo de ajuste (s)"] = round(r["mean_fit_time"][i], 1)
    filas.append(fila)

comparacion = pd.DataFrame(filas).sort_values("AUPRC validación", ascending=False).reset_index(drop=True)
comparacion.to_csv(f"{CARPETA_RESULTADOS}/comparacion_modelos.csv", index=False)
comparacion

In [ ]:
orden = comparacion.sort_values("AUPRC validación")

plt.figure(figsize=(7, 4))
plt.barh(orden["Modelo"], orden["AUPRC validación"], xerr=orden["IC 95 % (±)"], capsize=4)
plt.axvline(referencia_auprc, linestyle="--", color="gray", label="Modelo sin información")

plt.xlabel("AUPRC en validación (promedio ± IC 95 %)")
plt.ylabel("Modelo")
plt.title("Comparación de los modelos")
plt.legend()
plt.tight_layout()

plt.savefig(f"{CARPETA_RESULTADOS}/comparacion_modelos.png", dpi=200)
plt.show()

## 16. Hiperparámetros analizados

Resumen de las mallas de valores probadas para cada modelo. Esta tabla es la base de la tabla de hiperparámetros del reporte.

In [ ]:
filas = []
for nombre, malla in mallas_usadas.items():
    for parametro, valores in malla.items():
        filas.append({
            "Modelo": nombre,
            "Hiperparámetro": parametro.split("__")[-1],
            "Valores evaluados": ", ".join(str(v) for v in valores),
        })

tabla_hiperparametros = pd.DataFrame(filas)
tabla_hiperparametros.to_csv(f"{CARPETA_RESULTADOS}/hiperparametros_evaluados.csv", index=False)
tabla_hiperparametros

## 17. Interpretación y siguientes pasos

Completar con lo que se observó al ejecutar:

- Modelo con mejor AUPRC en validación y diferencia con los demás (¿los intervalos de confianza se solapan?):
- Modelos que muestran sobreajuste (AUPRC de entrenamiento mucho mayor que el de validación):
- Hiperparámetros que más influyen en cada modelo:
- Modelos más costosos en tiempo de ajuste:

Siguientes pasos:

- Elegir los dos mejores modelos y evaluarlos **una sola vez** en el conjunto de prueba, con intervalos de confianza (por ejemplo, por bootstrap).
- Reducción de dimensión sobre esos dos modelos. Como `V1` a `V28` ya son componentes PCA, no se aplica PCA sobre ellas; la estrategia se define en el notebook 03.
- Evaluar una partición temporal usando `Time`, ya que no hay identificador de titular para una validación agrupada.